## 4.3 HARQ 重传仿真

在上一节中，我们学习了 HARQ Chase Combining 的原理。本节先演示单次 HARQ 重传过程，再逐 SNR 扫描有无 HARQ 的 FER 与吞吐量对比。

本节学习大纲如下：

- HARQ 重传过程演示
- HARQ FER / 吞吐量 SNR 扫描

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── phy/
│   ├── mac_interface.py     <- mac_to_iq / iq_to_mac: MAC-PHY 转换
│   └── tx_pipeline.py       <- TxConfig: 发射参数配置 (MCS、CRC、导频间隔)
├── sim/
│   └── link_sim.py          <- sim_harq_link: HARQ FER/吞吐量 SNR 扫描
│                               _channel_impair: AWGN 加噪
```


---

### 1. HARQ 重传过程

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "2922,3035p"

以下演示 sim_harq_link() 的具体重传过程：

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 6.0
max_retries = 3

rng = np.random.default_rng(42)
cfg = TxConfig(
    frame_type=2, mcs_index=7, pid=0x123456,
    whitening_seed=0x52, crc_seed=0x555555,
    crc_len=24, ctrl_bits_len=28, pilot_interval=8,
)
payload = bytes(rng.integers(0, 256, 20, dtype=np.uint8))

# 首次传输
attempts = 1
iq = mac_to_iq(payload, cfg)
rx_iq = _channel_impair(iq, snr_db, "awgn", 6.0, 0.0, "none", cfg.sps, rng)
rx = iq_to_mac(rx_iq, cfg, len(payload))
success = rx.crc_ok and rx.mac_payload == payload
print(f"Attempt {attempts}: CRC={'OK' if success else 'FAIL'}")

# HARQ 重传循环
while not success and attempts <= max_retries:  #重传条件：接收到 NACK 并且重传次数小于最大重传次数
    attempts += 1
    iq = mac_to_iq(payload, cfg)
    rx_iq = _channel_impair(iq, snr_db, "awgn", 6.0, 0.0, "none", cfg.sps, rng)
    rx = iq_to_mac(rx_iq, cfg, len(payload))
    success = rx.crc_ok and rx.mac_payload == payload
    tag = "(HARQ success!)" if success else "(retransmit)"
    print(f"Attempt {attempts}: CRC={'OK' if success else 'FAIL'} {tag}")

print(f"\nResult: {'PASS' if success else 'FAIL'} after {attempts} tx")

将以上过程加入 SNR 扫描后封装成 sim_harq_link 函数，可执行以下代码查看具体函数：

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "2922,3037p"

---

### 2. HARQ FER / 吞吐量 SNR 扫描

In [ ]:
from nearlink_sdr.sim.link_sim import sim_harq_link
import matplotlib.pyplot as plt

result = sim_harq_link(snr_range_db=np.arange(-2, 14, 1), n_frames=200,
                       mcs_index=7, payload_size=20, max_retries=3, seed=42)

print(f"{'SNR':>5s}  {'FER(no)':>10s}  {'FER(HARQ)':>10s}  {'AvgTx':>6s}  {'TP(no)':>10s}  {'TP(HARQ)':>10s}")
for i, snr in enumerate(result["snr_db"]):
    print(f"{snr:5.0f}  {result['fer_no_harq'][i]:10.4f}  {result['fer_harq'][i]:10.4f}  "
          f"{result['avg_transmissions'][i]:6.2f}  {result['throughput_no_harq'][i]:10.4f}  "
          f"{result['throughput_harq'][i]:10.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(result["snr_db"], [max(f, 1e-4) for f in result["fer_no_harq"]], "o-", label="No HARQ")
ax1.semilogy(result["snr_db"], [max(f, 1e-4) for f in result["fer_harq"]], "s-", label=f"HARQ (max 3)")
ax1.set_xlabel("SNR (dB)"); ax1.set_ylabel("FER"); ax1.set_title("HARQ FER Improvement")
ax1.legend(); ax1.grid(True, which="both", ls="--", alpha=0.5)

ax2.plot(result["snr_db"], result["throughput_no_harq"], "o-", label="No HARQ")
ax2.plot(result["snr_db"], result["throughput_harq"], "s-", label="HARQ")
ax2.set_xlabel("SNR (dB)"); ax2.set_ylabel("Throughput (bit/symbol)"); ax2.set_title("HARQ Throughput")
ax2.legend(loc="upper left"); ax2.grid(True, ls="--", alpha=0.3)
ax2b = ax2.twinx()
ax2b.plot(result["snr_db"], result["avg_transmissions"], "^--", color="gray", alpha=0.6, label="Avg TX")
ax2b.set_ylabel("Avg Transmissions"); ax2b.legend(loc="upper right")
plt.tight_layout(); plt.show()

### 预期观察结果

从上方图表可以观察以下要点：

- **低 SNR 区（< 4 dB）**：有无 HARQ 的 FER 均接近 1，噪声太强，重传再多帧也无法成功解码。此区域内 AvgTx 接近 max_retries+1=4，每帧都用尽重传次数后放弃。
- **中 SNR 区（5-8 dB）**：HARQ 的 FER 曲线明显低于无 HARQ——重传给帧提供了额外的成功机会。观察 AvgTx 列，HARQ 成功帧的平均传输次数逐步降低，说明随着 SNR 改善，越来越少帧需要重传。
- **高 SNR 区（> 9 dB）**：两者 FER 均归零，AvgTx 趋近 1.0——几乎所有帧在首次传输即成功，重传机制不再被触发。HARQ 的收益仅在中 SNR 区有效。

---

## 课后实践

请补全下方 HARQ 重传循环中的 **2 处空缺**（每处一行代码），使程序能够在 SNR=7 dB 下正确执行重传直到成功或达到最大次数。

要求：

1. 补全 `while` 循环的重传条件
2. 补全每次重传后 `success` 状态的更新

完成后运行 `python harq_practice.py` 验证结果。

In [ ]:
%%writefile harq_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 7.0
max_retries = 3
rng = np.random.default_rng(42)
cfg = TxConfig(frame_type=2, mcs_index=7, pid=0x123456,
               whitening_seed=0x52, crc_seed=0x555555,
               crc_len=24, ctrl_bits_len=28, pilot_interval=8)
payload = bytes(rng.integers(0, 256, 20, dtype=np.uint8))

# 首次传输
attempts = 1
iq = mac_to_iq(payload, cfg)
rx_iq = _channel_impair(iq, snr_db, "awgn", 6.0, 0.0, "none", cfg.sps, rng)
rx = iq_to_mac(rx_iq, cfg, len(payload))
success = rx.crc_ok and rx.mac_payload == payload

# ====  补全 HARQ 重传循环  ====
while            :
    attempts += 1
    iq = mac_to_iq(payload, cfg)
    rx_iq = _channel_impair(iq, snr_db, "awgn", 6.0, 0.0, "none", cfg.sps, rng)
    rx = iq_to_mac(rx_iq, cfg, len(payload))
    success =            # 更新 success 状态

print(f"SNR={snr_db}dB: {'PASS' if success else 'FAIL'} after {attempts} tx")


执行以下命令进行编译并验证结果：


In [ ]:
!python harq_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/04.03_answer.txt
